# Manuscript-Ready Plots and Tables

Combines the per-analysis outputs of `X2_agent_analysis.ipynb` and
`X3_run_analysis.ipynb` into the final tables and the combined contrast-forest
figure used in the paper (`Appendix E`).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from plot_utils import _configure_fonts, SETTING_EXTENDED_MAP, OKABE_ITO, GRAPH_COLOR

_configure_fonts()

result_path = Path.cwd().parent / "data" / "analysis"

# The four outcomes reported throughout the paper: three agent-level
# (plasticity, directedness, outgoing influence) and one population-level
# (consensus change).
COL_PRETTY_NAMES = {
    "plasticity_tv": "Plasticity",
    "monotonicity": "Directedness",
    "influence_out_joint": "Outgoing Influence",
    "modal_consensus_change": "Consensus Change",
}

CONTRAST_LABELS = {
    "role_eff": "Role effect",
    "model_eff": "Specialization effect",
    "alignment": "Role--spec. alignment",
    "dissociation": "Composition effect",
}

GRAPH_LABELS = {"erdos-renyi": "ER", "watts-strogatz": "WS", "marginal": "Marginal"}


def fmt_p(p: float) -> str:
    if pd.isna(p):
        return ""
    return "<0.001" if p < 0.001 else f"{p:.3f}"

## Aggregated estimated marginal means (Table "agg-results")

Agent-level outcomes come from `X2_agent_analysis.ipynb`'s per-metric EMM
fits; the population-level consensus-change outcome comes from
`X3_run_analysis.ipynb`. Both the graph-type-pooled ("marginal") and
per-network-type EMMs are included.


In [ ]:
def _load_emm(col: str, agg_subdir: str) -> list[pl.DataFrame]:
    base = result_path / "aggregated" / agg_subdir
    marginal = (
        pl.read_csv(base / f"{col}_setting" / "emm.csv")
        .with_columns(pl.lit("marginal").alias("graph_type"), pl.lit(col).alias("metric"))
        .select(["setting", "emmean", "SE", "df", "lower.CL", "upper.CL", "graph_type", "metric"])
    )
    by_graph_type = (
        pl.read_csv(base / f"{col}_setting_graph_type" / "emm.csv")
        .with_columns(pl.lit(col).alias("metric"))
        .select(["setting", "emmean", "SE", "df", "lower.CL", "upper.CL", "graph_type", "metric"])
    )
    return [marginal, by_graph_type]


agg_res = []
for col in COL_PRETTY_NAMES:
    agg_subdir = "runs" if col == "modal_consensus_change" else "agent"
    agg_res.extend(_load_emm(col, agg_subdir))

metric_order = list(COL_PRETTY_NAMES.keys())
lfs_res = (
    pl.concat(agg_res)
    .rename({
        "setting": "Scenario", "emmean": "Estimate", "lower.CL": "CI low",
        "upper.CL": "CI high", "metric": "Outcome", "graph_type": "Network",
    })
    .with_columns(
        pl.col("Outcome").map_elements(lambda x: COL_PRETTY_NAMES.get(x, x), return_dtype=pl.String).alias("Outcome"),
        pl.col("Scenario").map_elements(lambda x: SETTING_EXTENDED_MAP.get(x, x), return_dtype=pl.String).alias("Scenario"),
        pl.col("Network").map_elements(lambda x: GRAPH_LABELS.get(x, x), return_dtype=pl.String).alias("Network"),
    )
    .select(
        "Scenario", "Outcome", "Network",
        pl.col("Estimate").round(2).alias("Estimate"),
        pl.col("CI low").round(2).alias("CI low"),
        pl.col("CI high").round(2).alias("CI high"),
        pl.col("SE").round(3).alias("SE"),
        pl.col("df").round(1).alias("df"),
    )
).to_pandas()

lfs_res["Scenario"] = pd.Categorical(lfs_res["Scenario"], categories=list(SETTING_EXTENDED_MAP.values()), ordered=True)
lfs_res["Outcome"] = pd.Categorical(lfs_res["Outcome"], categories=[COL_PRETTY_NAMES[o] for o in metric_order], ordered=True)
lfs_res["Network"] = pd.Categorical(lfs_res["Network"], categories=["ER", "WS", "Marginal"], ordered=True)
lfs_res.to_csv(result_path / "aggregated" / "agg_results.csv", index=False)

lfs_res = lfs_res.sort_values(["Outcome", "Scenario", "Network"]).set_index(["Outcome", "Scenario", "Network"])
lfs_res.head()


### LaTeX

In [ ]:
out_res = lfs_res.to_latex(
    index=True, caption="", multicolumn=True, multirow=True, escape=False,
    float_format="{:0.2f}".format, label="tab:agg-results",
)
out_res = out_res.replace("\\multirow[t]{", "\\multirow[c]{")
print(out_res)


## Contrasts

Combine the agent-level contrasts (`X2_agent_analysis.ipynb`) with the
run-level consensus-change contrasts (`X3_run_analysis.ipynb`).

In [ ]:
agent_contrast = pl.read_csv(result_path / "contrasts" / "agent" / "contrast_summary.csv")
run_contrast = pl.read_csv(result_path / "contrasts" / "runs" / "contrast_summary.csv")
contrasts = pd.concat([agent_contrast.to_pandas(), run_contrast.to_pandas()], ignore_index=True)
contrasts.to_csv(result_path / "contrasts" / "combined_contrasts.csv", index=False)
contrasts.sample(min(10, len(contrasts)))


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator


def plot_contrast_forest(summary, contrasts_order, outcomes_order, graph_types_order=("erdos-renyi", "watts-strogatz"), figsize=(14, 4.8)):
    """Forest plot with one panel per contrast and 3 estimates (marginal, ER, WS) per outcome."""
    triplet_specs = [
        ("marginal", None, "Marginal (pooled)", OKABE_ITO["blue"], 0.0),
        ("within", graph_types_order[0], "Erdos-Renyi", GRAPH_COLOR["erdos-renyi"], -0.23),
        ("within", graph_types_order[1], "Watts-Strogatz", GRAPH_COLOR["watts-strogatz"], 0.23),
    ]

    fig, axes = plt.subplots(1, len(contrasts_order), figsize=figsize, sharey=True, sharex=True, squeeze=False)
    axes = axes[0]
    y_positions = {o: i for i, o in enumerate(outcomes_order)}

    for panel_idx, (ax, contrast_name) in enumerate(zip(axes, contrasts_order)):
        sub_contrast = summary[summary["contrast"] == contrast_name]
        for y in range(len(outcomes_order)):
            ax.axhline(y + 0.5, color="0.88", linewidth=0.7, alpha=0.7, zorder=0)

        for outcome in outcomes_order:
            if outcome not in y_positions:
                continue
            y_base = y_positions[outcome]
            for ctype, gtype, _, color, y_offset in triplet_specs:
                mask = (sub_contrast["contrast_type"] == ctype) & (sub_contrast["outcome"] == outcome)
                mask &= sub_contrast["graph_type"].isna() if gtype is None else sub_contrast["graph_type"] == gtype
                sub = sub_contrast[mask]
                if sub.empty:
                    continue
                row = sub.iloc[0]
                is_sig = bool(row["sig"])
                ax.errorbar(
                    row["cohens_d"], y_base + y_offset,
                    xerr=[[row["cohens_d"] - row["cohens_d_lower"]], [row["cohens_d_upper"] - row["cohens_d"]]],
                    fmt="o", color=color, markerfacecolor=color if is_sig else "white", markeredgecolor=color,
                    capsize=3, markersize=3, alpha=1.0 if is_sig else 0.45, linewidth=1.5,
                )

        ax.axvline(0, color="k", linestyle="--", alpha=0.35, linewidth=1)
        panel_label = chr(ord("A") + panel_idx)
        ax.set_title(f"({panel_label}) {CONTRAST_LABELS.get(contrast_name, contrast_name)}", fontsize=13)
        ax.set_xlabel("Cohen's d", fontsize=12)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.5)
        ax.spines["bottom"].set_linewidth(1.5)
        ax.tick_params(axis="both", which="major", width=1.3, length=5)
        ax.xaxis.set_minor_locator(MultipleLocator(0.5))
        ax.tick_params(axis="x", which="minor", width=1.0, length=3)

    axes[0].set_yticks(range(len(outcomes_order)))
    axes[0].set_yticklabels([COL_PRETTY_NAMES.get(o, o) for o in outcomes_order], fontsize=12)

    legend_handles = [
        Line2D([0], [0], marker="o", color=c, markerfacecolor=c, markeredgecolor=c, linestyle="None", markersize=6, label=lbl)
        for _, _, lbl, c, _ in triplet_specs
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=3, frameon=False, fontsize=12, bbox_to_anchor=(0.5, -0.07))
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    return fig


contrasts_latex = pl.from_pandas(
    contrasts[contrasts["outcome"].isin(COL_PRETTY_NAMES.keys()) & contrasts["contrast_type"].isin(["marginal", "within", "interaction"])]
)


#### Marginal effects

In [ ]:
lfs = (
    contrasts_latex.filter(pl.col("contrast_type") == "marginal")
    .with_columns(
        pl.col("outcome").map_elements(lambda x: COL_PRETTY_NAMES.get(x, x), return_dtype=pl.String).alias("Outcome"),
        pl.col("contrast").map_elements(lambda x: CONTRAST_LABELS.get(x, x), return_dtype=pl.String).alias("Contrast"),
    )
    .select(
        "Outcome", "Contrast",
        pl.col("cohens_d").round(3).alias("Cohen's d"),
        pl.col("cohens_d_lower").round(3).alias("d CI lower"),
        pl.col("cohens_d_upper").round(3).alias("d CI upper"),
        pl.col("SE").round(3).cast(pl.String).alias("SE"),
        pl.col("t").round(3),
        pl.col("p").map_elements(fmt_p, return_dtype=pl.String).alias("p"),
        pl.col("df").floor().cast(pl.Int32).alias("df"),
        pl.when(pl.col("sig")).then(pl.lit("*")).otherwise(pl.lit("")).alias("Sig"),
    )
).sort(["Outcome", "Contrast"]).to_pandas()

lfs["Contrast"] = pd.Categorical(lfs["Contrast"], categories=list(CONTRAST_LABELS.values()), ordered=True)
lfs["Outcome"] = pd.Categorical(lfs["Outcome"], categories=[COL_PRETTY_NAMES[o] for o in COL_PRETTY_NAMES], ordered=True)
lfs = lfs.sort_values(["Outcome", "Contrast"]).set_index(["Outcome", "Contrast"])

marginal_latex = lfs.to_latex(
    index=True, caption="Marginal contrasts on key outcomes.", multicolumn=True,
    multirow=True, escape=False, float_format="{:0.2f}".format, label="tab:marginal-contrasts",
)
marginal_latex = marginal_latex.replace("\\multirow[t]{", "\\multirow[c]{")
print(marginal_latex)


#### Conditional and interaction effects

In [ ]:
network_expr = (
    pl.when(pl.col("contrast_type") == "interaction")
    .then(pl.lit("Between"))
    .otherwise(pl.col("graph_type").replace_strict(GRAPH_LABELS, default=pl.col("graph_type")))
    .alias("Network")
)

lfs = (
    contrasts_latex
    .filter(pl.col("contrast_type").is_in(["within", "interaction"]))
    .with_columns(
        pl.col("outcome").map_elements(lambda x: COL_PRETTY_NAMES.get(x, x), return_dtype=pl.String).alias("Outcome"),
        pl.col("contrast").map_elements(lambda x: CONTRAST_LABELS.get(x, x), return_dtype=pl.String).alias("Contrast"),
        network_expr,
    )
    .select(
        "Outcome", "Contrast", "Network",
        pl.col("cohens_d").round(3).alias("Cohen's d"),
        pl.col("cohens_d_lower").round(3).alias("d CI lower"),
        pl.col("cohens_d_upper").round(3).alias("d CI upper"),
        pl.col("SE").round(3).cast(pl.String).alias("SE"),
        pl.col("t").round(3),
        pl.col("p").map_elements(fmt_p, return_dtype=pl.String).alias("p"),
        pl.col("df").cast(pl.Int32).alias("df"),
        pl.when(pl.col("sig")).then(pl.lit("*")).otherwise(pl.lit("")).alias("Sig"),
    )
).to_pandas()

lfs["Contrast"] = pd.Categorical(lfs["Contrast"], categories=list(CONTRAST_LABELS.values()), ordered=True)
lfs["Outcome"] = pd.Categorical(lfs["Outcome"], categories=[COL_PRETTY_NAMES[o] for o in COL_PRETTY_NAMES], ordered=True)
lfs["Network"] = pd.Categorical(lfs["Network"], categories=["ER", "WS", "Between"], ordered=True)

# Group by contrast so ER/WS/Between sit together per contrast.
lfs = lfs.sort_values(["Outcome", "Network", "Contrast"]).set_index(["Outcome", "Network", "Contrast"])

cond_latex = lfs.to_latex(
    index=True,
    caption="Within-network and between-network contrasts on key outcomes. "
            "The \\emph{Between} row reports the topology$\\times$contrast interaction (ER$-$WS).",
    multicolumn=True, multirow=True, escape=False, float_format="{:0.2f}".format,
    label="tab:within-between-contrasts",
)
cond_latex = cond_latex.replace("\\multirow[t]{", "\\multirow[c]{")
print(cond_latex)


In [ ]:
fig = plot_contrast_forest(
    contrasts[contrasts["outcome"].isin(COL_PRETTY_NAMES.keys())],
    contrasts_order=["role_eff", "model_eff", "alignment", "dissociation"],
    outcomes_order=list(COL_PRETTY_NAMES.keys()),
    figsize=(11, 3),
)
fig.savefig(result_path / "contrasts" / "contrast_forest.pdf", dpi=300, bbox_inches="tight")
plt.show()
